# 01.9 — Stereocentre Analysis (per endpoint, from SDF)

Follow-up to the stereoisomer-pair exclusion rule (`exclude_stereoisomer_pairs`, ADR-confirmed
0 excluded pairs on the public CSV). That check answers *"are there duplicate stereoisomer pairs
we need to filter"* — answer: no. This notebook answers a different question: *"how much of the
data we actually model carries unassigned stereochemistry, and does that vary by endpoint?"*

**Source**: the six per-endpoint SDF files (`data/sdfs/ADME_*.sdf`), loaded and standardized the
same way as the modelling pipeline in `01.5_adme_biogen_public_recreation.ipynb` (§2.4) — not the
flat CSV. This matters for PPB_H/PPB_R, which are ChEMBL-augmented in the SDFs (1795/876 rows)
vs sparse in the CSV (~170 rows).

**What we count**: for each molecule, RDKit's `FindMolChiralCenters(includeUnassigned=True)`
returns every potential stereocentre and whether it's assigned (R/S) or left unassigned (`?`) in
the source structure. A molecule with an unassigned stereocentre is one where FCFP4/ECFP4
fingerprints cannot distinguish which stereoisomer was actually tested.

In [ ]:
import sys
sys.path.insert(0, '..')

import contextlib
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import FindMolChiralCenters
from rdkit.Chem import SDMolSupplier

from src.preprocessing import standardize

DATA_RAW  = Path('../data/raw')
DATA_PROC = Path('../data/processed')
FIGURES   = Path('../figures')

SDF_DIR = DATA_RAW.parent / 'sdfs'
LOG_FILE = DATA_PROC / 'stereo_analysis_standardization.log'

SDF_FILES = {
    'HLM':   SDF_DIR / 'ADME_HLM.sdf',
    'MDR1':  SDF_DIR / 'ADME_MDR1_ER.sdf',
    'RLM':   SDF_DIR / 'ADME_RLM.sdf',
    'SOL':   SDF_DIR / 'ADME_Sol.sdf',
    'PPB_H': SDF_DIR / 'ADME_hPPB.sdf',
    'PPB_R': SDF_DIR / 'ADME_rPPB.sdf',
}

ENDPOINTS = {
    'HLM':   'LOG HLM_CLint (mL/min/kg)',
    'MDR1':  'LOG MDR1-MDCK ER (B-A/A-B)',
    'RLM':   'LOG RLM_CLint (mL/min/kg)',
    'SOL':   'LOG SOLUBILITY PH 6.8 (ug/mL)',
    'PPB_H': 'LOG PLASMA PROTEIN BINDING (HUMAN) (% unbound)',
    'PPB_R': 'LOG PLASMA PROTEIN BINDING (RAT) (% unbound)',
}

print('Setup complete')
for ep, path in SDF_FILES.items():
    print(f'{ep}: {path.exists()}')

## 1. Load and standardize each endpoint's SDF

Same loading + standardization + dedup-by-canonical-SMILES logic as `01.5` §2.4, so the molecule
sets here match what the models are actually trained on.

In [ ]:
ep_dfs = {}
with open(LOG_FILE, 'w') as log_fh, contextlib.redirect_stderr(log_fh):
    for ep, path in SDF_FILES.items():
        col = ENDPOINTS[ep]
        rows = []
        for mol in SDMolSupplier(str(path)):
            if mol is None:
                continue
            value = float(mol.GetProp(col))
            mol = standardize(mol)
            rows.append({
                'can_smi': Chem.MolToSmiles(mol),
                'mol':     mol,
                col:       value,
            })
        ep_dfs[ep] = pd.DataFrame(rows).drop_duplicates('can_smi').reset_index(drop=True)
        print(f'{ep}: {len(ep_dfs[ep])} molecules loaded')

print(f'\nRDKit logs written to: {LOG_FILE}')

## 2. Count stereocentres per molecule

`FindMolChiralCenters(includeUnassigned=True, useLegacyImplementation=False)` returns every
tetrahedral stereocentre RDKit can find, whether or not the source structure specifies R/S.

In [ ]:
def stereo_counts(mol):
    """Return (n_total, n_defined, n_undefined) stereocentres for a mol."""
    centers = FindMolChiralCenters(mol, includeUnassigned=True, useLegacyImplementation=False)
    n_undefined = sum(1 for _, tag in centers if tag == '?')
    n_defined = len(centers) - n_undefined
    return len(centers), n_defined, n_undefined


for ep, df_ep in ep_dfs.items():
    counts = df_ep['mol'].apply(stereo_counts)
    df_ep['n_stereocenters'] = counts.apply(lambda c: c[0])
    df_ep['n_defined'] = counts.apply(lambda c: c[1])
    df_ep['n_undefined'] = counts.apply(lambda c: c[2])

print('Stereocentre counts computed for all endpoints')

## 3. Per-endpoint summary

For each endpoint's modelled molecule set: how many compounds carry a stereocentre at all, and
of those, how many are left unassigned (the fingerprint-blind-spot cases).

In [ ]:
summary_rows = []
for ep, df_ep in ep_dfs.items():
    n_mols = len(df_ep)
    n_any_center = (df_ep['n_stereocenters'] > 0).sum()
    n_any_undefined = (df_ep['n_undefined'] > 0).sum()
    summary_rows.append({
        'endpoint': ep,
        'N modelled': n_mols,
        'N with stereocentre': n_any_center,
        '% with stereocentre': round(100 * n_any_center / n_mols, 1),
        'N with unassigned stereocentre': n_any_undefined,
        '% with unassigned stereocentre': round(100 * n_any_undefined / n_mols, 1),
        'total stereocentres': int(df_ep['n_stereocenters'].sum()),
        'total unassigned': int(df_ep['n_undefined'].sum()),
    })

summary_df = pd.DataFrame(summary_rows).set_index('endpoint')
summary_df = summary_df.sort_values('% with unassigned stereocentre', ascending=False)
summary_df

## 4. Whole-dataset figure, for comparison

The union of unique molecules across all six endpoint SDFs (mirrors `df_sdf` in `01.5`) — the
SDF-sourced equivalent of the earlier CSV-only figure (3521 compounds, 210 with an unassigned
stereocentre).

In [ ]:
all_mols = (
    pd.concat([d[['can_smi', 'mol', 'n_stereocenters', 'n_defined', 'n_undefined']] for d in ep_dfs.values()])
    .drop_duplicates('can_smi')
    .reset_index(drop=True)
)

n_mols = len(all_mols)
n_any_center = (all_mols['n_stereocenters'] > 0).sum()
n_any_undefined = (all_mols['n_undefined'] > 0).sum()

print(f'Unique molecules across all endpoint SDFs: {n_mols}')
print(f'With >=1 stereocentre: {n_any_center} ({100 * n_any_center / n_mols:.1f}%)')
print(f'With >=1 UNDEFINED stereocentre: {n_any_undefined} ({100 * n_any_undefined / n_mols:.1f}%)')
print(f'Total stereocentres: {int(all_mols["n_stereocenters"].sum())} '
      f'(defined={int(all_mols["n_defined"].sum())}, undefined={int(all_mols["n_undefined"].sum())})')

## 5. Does one endpoint carry more unassigned stereochemistry than the rest?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

plot_df = summary_df.sort_values('% with unassigned stereocentre', ascending=True)
bars = ax.barh(plot_df.index, plot_df['% with unassigned stereocentre'], color='steelblue')
ax.bar_label(bars, fmt='%.1f%%', padding=3)

overall_pct = 100 * n_any_undefined / n_mols
ax.axvline(overall_pct, color='grey', linestyle='--', linewidth=1)
ax.text(overall_pct, -0.7, f'  all-endpoints union: {overall_pct:.1f}%', color='grey', va='top', fontsize=9)

ax.set_xlabel('% of modelled compounds with >=1 unassigned stereocentre')
ax.set_title('Unassigned stereochemistry by endpoint (SDF-sourced, standardized)')
ax.set_xlim(0, max(plot_df['% with unassigned stereocentre'].max(), overall_pct) * 1.25)
plt.tight_layout()
plt.savefig(FIGURES / 'stereo_unassigned_by_endpoint.png', dpi=150)
plt.show()

## 6. Stereoisomer-pair exclusion, re-checked per endpoint

The original question this all started from: `exclude_stereoisomer_pairs` found 0 excluded pairs
on the flat CSV (3521 compounds, all endpoints pooled). Re-run the same flat-SMILES duplicate
check per endpoint's modelled set, since PPB_H/PPB_R draw from a different (larger, ChEMBL-
augmented) pool than the CSV check covered.

In [ ]:
from src.cleaning import exclude_stereoisomer_pairs

for ep, df_ep in ep_dfs.items():
    col = ENDPOINTS[ep]
    _, excluded = exclude_stereoisomer_pairs(df_ep, 'can_smi', [col], fold_threshold=3)
    print(f'{ep}: {len(excluded)} compounds excluded ({len(excluded) // 2} stereoisomer pairs)')

## 7. Open item — not actioned

PPB_H has 3 stereoisomer pairs (6 compounds) that trigger the >3-fold exclusion rule — a real gap
in the earlier "0 excluded pairs" claim, which was checked against the CSV only and doesn't cover
PPB's ChEMBL-augmented SDF pool. HLM/MDR1/RLM/SOL/PPB_R all still show 0.

**Not fixed here**: PPB isn't part of the current report/blog-post scope, and `exclude_stereoisomer_pairs`
is not yet wired into the PPB_H modelling path in `01.5`/`01.6` regardless. Flagging as a known
correctness gap to close if/when PPB modelling work resumes — apply `exclude_stereoisomer_pairs`
to `ep_dfs['PPB_H']` before featurization, matching how HLM/MDR1/RLM/SOL are already (implicitly)
clean of this issue.